# Test `AnnotationCollection.push_annotations` with pygit2

This notebook tests the pygit2-based implementation of `AnnotationCollection.push_annotations()`
against the empty `demo_for_pygit2` project on CATMA's GitLab backend.

The workflow is:

1. Load the (empty) project clone that is stored in `user_projects/`.
2. Add a document and a tagset to it (copied from the bundled demo project).
3. Create a new annotation collection.
4. Add annotations to the collection.
5. Push everything to the CATMA GitLab backend using `AnnotationCollection.push_annotations()`.


In [ ]:
# load local gitma
!pip install setuptools --upgrade
!pip install -e .
!pip show gitma


In [ ]:
## Run tests

!python -m unittest discover -s tests -p "test*.py"

In [ ]:
import json
import os
import shutil
import subprocess
import uuid

from gitma import CatmaProject

In [ ]:
# TODO: Replace with your own CATMA GitLab access token.
# Generate one at https://git.catma.de/-/profile/personal_access_tokens with the "api" scope.
gitlab_access_token = "put your key here"  # with write rights

project_name = "demo_for_pygit2"
projects_directory = "user_projects/"

## Load the project

The `demo_for_pygit2` project was already cloned into `user_projects/`.
It is empty, so no documents, tagsets or annotation collections are loaded.


In [ ]:
project = CatmaProject(
    project_name=project_name,
    projects_directory=projects_directory,
    gitlab_access_token=gitlab_access_token,
)
print(project)

DEMO_PROJECT = (
    "demo/projects/CATMA_9385E190-13CD-44BE-8A06-32FA95B7EEFA_GitMA_Demo_Project"
)
project_dir = f"{projects_directory}/{project.uuid}"

## Add a document and a tagset

Writing annotations requires a document and a tagset.
We copy the document ("The Metamorphosis") and the tagset ("demo_tagset") from the bundled
demo project into the empty `demo_for_pygit2` project.


In [ ]:
for subdir in ("documents", "tagsets"):
    shutil.copytree(
        os.path.join(DEMO_PROJECT, subdir),
        os.path.join(project_dir, subdir),
    )
print("Copied documents/ and tagsets/ into", project_dir)

## Create an annotation collection

An annotation collection is a directory `collections/<UUID>/` containing a `header.json`.
The `sourceDocumentId` points to the document that the collection annotates.


In [ ]:
text_uuid = os.listdir(f"{project_dir}/documents")[0]

ac_name = "test_ac_pygit2_1"
ac_uuid = f"C_{uuid.uuid4().hex.upper()}"
ac_dir = f"{project_dir}/collections/{ac_uuid}"
os.makedirs(f"{ac_dir}/annotations")

with open(f"{ac_dir}/header.json", "w", encoding="utf-8", newline="") as header_output:
    json.dump(
        {
            "author": None,
            "description": "annotation collection created with gitma - second run",
            "forkedFromCommitURL": None,
            "name": ac_name,
            "publisher": None,
            "responsibleUser": "GitMA_DemoUser_2",
            "sourceDocumentId": text_uuid,
        },
        header_output,
        indent=2,
    )

print(f'Created annotation collection "{ac_name}" with UUID {ac_uuid}.')

## Reload the project

Reload the project so that the copied document, tagset and the new annotation collection get loaded.


In [ ]:
project = CatmaProject(
    project_name=project_name,
    projects_directory=projects_directory,
    gitlab_access_token=gitlab_access_token,
)
print(project)

text = project.text_dict["The Metamorphosis"]
ac = project.ac_dict[ac_name]
print(ac)

## Add annotations

Annotations are added to the collection with `CatmaProject.write_annotation_json()`.


In [ ]:
project.write_annotation_json(
    text_title=text.title,
    annotation_collection_name=ac.name,
    tagset_name="demo_tagset",
    tag_name="non_event",
    start_points=[20],
    end_points=[151],
    property_annotations={},
    author="GitMA_DemoUser",
)
project.write_annotation_json(
    text_title=text.title,
    annotation_collection_name=ac.name,
    tagset_name="demo_tagset",
    tag_name="change_of_state",
    start_points=[450],
    end_points=[560],
    property_annotations={"representation_type": ["narrator_speech"]},
    author="GitMA_DemoUser",
)

# Load project - skip here if you have populated the project already

Reload the project and inspect the annotations that were just written.


In [ ]:
project = CatmaProject(
    project_name=project_name,
    projects_directory=projects_directory,
    gitlab_access_token=gitlab_access_token,
)

ac_name = "test_ac_pygit2_1"

ac = project.ac_dict[ac_name]
print(f'Found {len(ac.annotations)} annotations in collection "{ac.name}":')
for annotation in ac.annotations:
    print(annotation)

## Push the annotations

`AnnotationCollection.push_annotations()` stages all changes, creates a commit and pushes it
to the CATMA GitLab backend with pygit2.


In [ ]:
ac.push_annotations(commit_message="add annotations via pygit2")

In [ ]:
subprocess.run(["git", "-C", project_dir, "log", "--oneline", "-3"])
subprocess.run(["git", "-C", project_dir, "status", "--short"])